# โปรแกรมแก้ระบบสมการเชิงเส้น (Linear Algebra Solver)

รองรับ:
- Gauss Elimination with Pivoting
- Gauss-Jordan Elimination
- LU Factorization
- Inverse Matrix (Gauss-Jordan)

พร้อมตรวจสอบกรณี:
- มีคำตอบเดียว
- ไม่มีคำตอบ
- คำตอบไม่เป็นเอกลักษณ์
- ไม่มี Inverse


In [ ]:
import numpy as np

## ฟังก์ชันรับข้อมูล

In [ ]:

def input_matrix_only():
    n = int(input("กรอกขนาดเมทริกซ์ (n): "))
    print("กรอก Matrix A ทีละแถว")
    A = []
    for i in range(n):
        A.append(list(map(float, input(f"A แถวที่ {i+1}: ").split())))
    return np.array(A, float)

def input_linear_system():
    n = int(input("กรอกจำนวนสมการ (n): "))
    print("กรอก Matrix A ทีละแถว")
    A = []
    for i in range(n):
        A.append(list(map(float, input(f"A แถวที่ {i+1}: ").split())))
    print("กรอกเวกเตอร์ b")
    b = list(map(float, input("b: ").split()))
    return np.array(A, float), np.array(b, float)


## วิธีที่ 1: Gauss Elimination with Pivoting

In [ ]:

def gauss_elimination_pivot(A, b):
    n = len(b)
    Aug = np.hstack((A, b.reshape(-1, 1)))

    for i in range(n):
        max_row = i
        for k in range(i+1, n):
            if abs(Aug[k, i]) > abs(Aug[max_row, i]):
                max_row = k
        Aug[[i, max_row]] = Aug[[max_row, i]]

        if abs(Aug[i, i]) < 1e-12:
            if abs(Aug[i, -1]) > 1e-12:
                return None, "ไม่มีคำตอบ (สมการขัดแย้ง)"
            else:
                return None, "มีคำตอบไม่เป็นเอกลักษณ์"

        for j in range(i+1, n):
            factor = Aug[j, i] / Aug[i, i]
            Aug[j] -= factor * Aug[i]

    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (Aug[i, -1] - np.dot(Aug[i, i+1:], x[i+1:])) / Aug[i, i]

    return x, "มีคำตอบเดียว"


## วิธีที่ 2: Gauss-Jordan Elimination

In [ ]:

def gauss_jordan_elimination(A, b):
    n = len(b)
    Aug = np.hstack((A, b.reshape(-1, 1)))

    for i in range(n):
        if abs(Aug[i, i]) < 1e-12:
            if abs(Aug[i, -1]) > 1e-12:
                return None, "ไม่มีคำตอบ (สมการขัดแย้ง)"
            else:
                return None, "มีคำตอบไม่เป็นเอกลักษณ์"

        Aug[i] /= Aug[i, i]

        for j in range(n):
            if j != i:
                Aug[j] -= Aug[j, i] * Aug[i]

    return Aug[:, -1], "มีคำตอบเดียว"


## วิธีที่ 3: LU Factorization

In [ ]:

def lu_factorization(A, b):
    n = len(b)
    L = np.zeros((n, n))
    U = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):
            U[i, j] = A[i, j] - sum(L[i, k] * U[k, j] for k in range(i))

        if abs(U[i, i]) < 1e-12:
            return None, "ไม่สามารถทำ LU ได้ (pivot = 0)"

        L[i, i] = 1
        for j in range(i+1, n):
            L[j, i] = (A[j, i] - sum(L[j, k] * U[k, i] for k in range(i))) / U[i, i]

    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - sum(L[i, j] * y[j] for j in range(i))

    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (y[i] - sum(U[i, j] * x[j] for j in range(i+1, n))) / U[i, i]

    return x, "มีคำตอบเดียว"


## วิธีที่ 4: Inverse Matrix (Gauss-Jordan)

In [ ]:

def inverse_matrix(A):
    n = len(A)
    Aug = np.hstack((A, np.eye(n)))

    for i in range(n):
        if abs(Aug[i, i]) < 1e-12:
            return None, "ไม่มี Inverse (determinant = 0)"

        Aug[i] /= Aug[i, i]

        for j in range(n):
            if j != i:
                Aug[j] -= Aug[j, i] * Aug[i]

    return Aug[:, n:], "มี Inverse"


## โปรแกรมหลัก

In [ ]:

while True:
    print("\n===== เมนูหลัก =====")
    print("1. แก้ Ax = b")
    print("2. หา Inverse Matrix")
    print("0. ออก")
    c = int(input("เลือก: "))

    if c == 0:
        print("สิ้นสุดโปรแกรม")
        break

    elif c == 1:
        A, b = input_linear_system()
        print("\nเลือกวิธี")
        print("1. Gauss Elimination")
        print("2. Gauss-Jordan")
        print("3. LU Factorization")
        m = int(input("เลือกวิธี: "))

        if m == 1:
            x, msg = gauss_elimination_pivot(A, b)
        elif m == 2:
            x, msg = gauss_jordan_elimination(A, b)
        elif m == 3:
            x, msg = lu_factorization(A, b)
        else:
            print("เลือกไม่ถูกต้อง")
            continue

        print(msg)
        if x is not None:
            print("x =", x)

    elif c == 2:
        A = input_matrix_only()
        Ainv, msg = inverse_matrix(A)
        print(msg)
        if Ainv is not None:
            print("A^-1 =")
            print(Ainv)
